# Day 3 — Baseline Modelling

## Objective
Build and evaluate baseline models to predict **Final Stable Dose (mg)** using
genetic, demographic, clinical, and lifestyle features.

These models establish a performance benchmark for later improvement
and ensure modelling choices are defensible and reproducible.

In [1]:
# ----------------------------
# Core libraries
# ----------------------------
import pandas as pd
import numpy as np

# Modelling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Utilities
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os
os.getcwd()

'C:\\Users\\caspe\\OneDrive\\Documents\\genexahealth\\genexahealth-data\\notebooks'

In [3]:
# ----------------------------
# Load prepared dataset
# ----------------------------
DATA_PATH = Path("../data/curated/merged_patient_dataset_prepared.csv")
df = pd.read_csv(DATA_PATH)

df.shape

(50000, 25)

In [4]:
# ----------------------------
# Define target variable
# ----------------------------
TARGET = "Final_Stable_Dose_mg"

y = df[TARGET]

# ----------------------------
# Drop identifiers and target from features
# ----------------------------
X = df.drop(columns=[
    TARGET,
    "patient_id",          # identifier, not predictive
    "Adverse_Event"        # keep flag, not raw text
])

X.head()

,CYP2C9,VKORC1,CYP4F2,Age,Sex,Weight_kg,Height_cm,Ethnicity,Hypertension,Diabetes,...,Amiodarone,Antibiotics,Aspirin,Statins,Alcohol_Intake,Smoking_Status,Diet_VitK_Intake,INR_Stabilization_Days,Time_in_Therapeutic_Range_Pct,Adverse_Event_Flag
0,*1/*3,A/G,C/C,64,F,88,194,Caucasian,0,0,...,0,0,1,0,Moderate,Non-smoker,Low,5,66.8,0
1,*1/*1,A/G,C/C,50,M,101,175,Other,1,0,...,0,0,0,1,Moderate,Non-smoker,Low,8,72.4,0
2,*1/*1,A/G,C/T,66,F,85,162,Asian,0,0,...,0,0,0,1,Heavy,Non-smoker,Medium,7,58.0,0
3,*1/*2,G/G,C/T,58,M,83,178,African American,1,0,...,0,0,0,0,Light,Non-smoker,Low,6,77.9,0
4,*1/*1,G/G,C/T,61,F,75,194,Caucasian,1,0,...,0,0,0,0,Light,Non-smoker,High,7,70.6,0


In [5]:
# ----------------------------
# Identify numeric & categorical columns
# ----------------------------
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_features, categorical_features

(['Age',
  'Weight_kg',
  'Height_cm',
  'Hypertension',
  'Diabetes',
  'Chronic_Kidney_Disease',
  'Heart_Failure',
  'Amiodarone',
  'Antibiotics',
  'Aspirin',
  'Statins',
  'INR_Stabilization_Days',
  'Time_in_Therapeutic_Range_Pct',
  'Adverse_Event_Flag'],
 ['CYP2C9',
  'VKORC1',
  'CYP4F2',
  'Sex',
  'Ethnicity',
  'Alcohol_Intake',
  'Smoking_Status',
  'Diet_VitK_Intake'])

In [6]:
# ----------------------------
# Preprocessing
# ----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [7]:
# ----------------------------
# Train / test split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
# ----------------------------
# Linear Regression pipeline
# ----------------------------
lr_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LinearRegression())
    ]
)

# Train
lr_model.fit(X_train, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test)

# Evaluate
lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = mean_squared_error(y_test, y_pred_lr, squared=False)
lr_r2 = r2_score(y_test, y_pred_lr)

lr_mae, lr_rmse, lr_r2

(0.5132653573608398, 0.6657574213187629, 0.831334533167443)

In [9]:
# ----------------------------
# Random Forest pipeline
# ----------------------------
rf_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Train
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)

# Evaluate
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = mean_squared_error(y_test, y_pred_rf, squared=False)
rf_r2 = r2_score(y_test, y_pred_rf)

rf_mae, rf_rmse, rf_r2

(0.4057147, 0.5648253883281098, 0.8785989174954425)

In [10]:
# ----------------------------
# Compare models
# ----------------------------
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [lr_mae, rf_mae],
    "RMSE": [lr_rmse, rf_rmse],
    "R2": [lr_r2, rf_r2]
})

results

,Model,MAE,RMSE,R2
0,Linear Regression,0.513265,0.665757,0.831335
1,Random Forest,0.405715,0.564825,0.878599
